# Immortalite One — T4 quality-neutral throughput roadmap

Runs the quality-neutral A0 profiling check, worker × concurrency matrix, and central-inference stage comparison against one real checkpoint. Results append to JSONL on Google Drive; `metrics.csv` is never read or modified.

Use a T4 GPU runtime. Run cells top to bottom. Every production lane preserves 150 simulations, root-Q targets, temperature 4.0 for 10 plies, draw semantics, resignation off, and the same production Syzygy files.

In [ ]:
# 1. Clone/update and build the native extension without replacing Colab's CUDA torch.
import os
%cd /content
if not os.path.exists('/content/immortalite-one'):
    !git clone https://github.com/carlo-wong/immortalite-one.git
%cd /content/immortalite-one
!git pull --quiet
!apt-get install -q -y build-essential ninja-build python3-dev
!pip install -q "cmake>=3.26,<4.0" ninja pybind11 scikit-build-core
!pip install -q python-chess numpy tqdm
!pip install -q -e . --no-deps

import subprocess
import torch
from engine import _native

assert torch.cuda.is_available(), 'Enable a GPU runtime before benchmarking.'
GPU_NAME = torch.cuda.get_device_name(0)
assert 'T4' in GPU_NAME, f'This notebook is controlled for T4; found {GPU_NAME!r}.'
print('git', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', GPU_NAME)
print('native', _native.version())

In [ ]:
# 2. Mount Drive and identify immutable benchmark inputs/artifacts.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import shutil

CKPT_DIR = Path('/content/drive/MyDrive/immortalite_zero_checkpoints')
CHECKPOINT = CKPT_DIR / 'latest.pt'
ARTIFACT_DIR = CKPT_DIR / 'phase_a_benchmarks'
OUTPUT = ARTIFACT_DIR / 'benchmark_throughput.jsonl'
TB_DRIVE = CKPT_DIR / 'syzygy345'
SYZYGY = Path('/content/syzygy345')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert CHECKPOINT.is_file(), f'Missing production checkpoint: {CHECKPOINT}'
assert TB_DRIVE.is_dir(), f'Missing production Syzygy directory: {TB_DRIVE}'

SYZYGY.mkdir(parents=True, exist_ok=True)
for source in TB_DRIVE.glob('*.rtbw'):
    destination = SYZYGY / source.name
    if not destination.exists():
        shutil.copy2(source, destination)
rtbw_files = list(SYZYGY.glob('*.rtbw'))
assert len(rtbw_files) == 145, f'Expected 145 Syzygy WDL files, found {len(rtbw_files)}'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('checkpoint', CHECKPOINT)
print('checkpoint sha256', sha256(CHECKPOINT))
print('output', OUTPUT)
print('syzygy', SYZYGY, len(rtbw_files), 'files')

## A0 — correctness and profiling overhead

This lane disables native root noise and move exploration only to make off/on fingerprints comparable. It uses the production checkpoint and 4 × 32 total concurrency. The runner fails if fingerprints differ or if overhead is at least 2% median / 3% p90.

In [ ]:
# 3. A0: one full-shape warm-up per mode, then five randomized paired trials.
# A0 measures profiling overhead on direct batching so native root-noise fingerprints
# remain comparable; central inference is compared separately below.
import subprocess

base = [
    'python', 'scripts/bench_throughput.py',
    '--checkpoint', str(CHECKPOINT),
    '--output', str(OUTPUT),
    '--device', 'cuda',
    '--workers', '4',
    '--concurrency', '128',
    '--games', '128',
    '--sims', '150',
    '--max-moves', '200',
    '--repeats', '5',
    '--syzygy-path', str(SYZYGY),
    '--central-inference', 'off',
]
subprocess.run(base + ['--a0-only'], check=True)

## A1/A2 — warm production matrix

Each worker configuration gets fresh processes, one cold observation, one full production-shape warm-up, then five measured 128-game trials. Configuration order is deterministic-randomized. With CUDA, multi-worker cells use central inference by default; one-worker cells use direct batching. Production root noise and move sampling remain enabled.

In [ ]:
# 4. Production default and worker × total-concurrency sweep.
matrix = [
    'python', 'scripts/bench_throughput.py',
    '--checkpoint', str(CHECKPOINT),
    '--output', str(OUTPUT),
    '--device', 'cuda',
    '--workers', '1,2,4',
    '--concurrency', '32,64,96,128',
    '--games', '128',
    '--sims', '150',
    '--max-moves', '200',
    '--warmups', '1',
    '--repeats', '5',
    '--syzygy-path', str(SYZYGY),
    '--central-inference', 'auto',
]
subprocess.run(matrix, check=True)
print('persisted', OUTPUT, OUTPUT.stat().st_size, 'bytes')

In [ ]:
# 5. Compact median ranking; the JSONL remains the source artifact.
import json
import statistics
from collections import defaultdict

rows = [json.loads(line) for line in OUTPUT.read_text().splitlines() if line.strip()]
production = [
    row for row in rows
    if row.get('variant', '').startswith('production-') and row.get('kind') == 'trial'
]
by_cell = defaultdict(list)
for row in production:
    recipe = row['recipe']
    by_cell[(row['variant'], recipe['workers'], recipe['total_concurrency'])].append(
        row['result']['seconds_per_game']
    )
ranking = sorted(
    (
        statistics.median(values),
        variant,
        workers,
        concurrency,
        len(values),
    )
    for (variant, workers, concurrency), values in by_cell.items()
)
for seconds_per_game, variant, workers, concurrency, trials in ranking:
    print(
        f'{seconds_per_game:8.3f} s/game  '
        f'{3600 / seconds_per_game:8.1f} games/hour  '
        f'{variant} workers={workers} concurrency={concurrency} n={trials}'
    )

In [ ]:
# 6. Record the supported throughput capability surface with this artifact set.
import json
from engine import _native
from engine import inference
from engine.network import CudaBatchExecutor, NetEvaluator

capabilities = {
    'evaluate_legal': hasattr(NetEvaluator, 'evaluate_legal'),
    'GameActorBatch': hasattr(_native, 'GameActorBatch'),
    'CudaBatchExecutor': CudaBatchExecutor is not None,
    'central_inference_module': hasattr(inference, 'CentralInferenceBroker'),
}
capability_row = {
    'schema_version': 1,
    'kind': 'capabilities',
    'variant': 'capability-surface',
    'checkpoint_sha256': sha256(CHECKPOINT),
    'capabilities': capabilities,
}
with OUTPUT.open('a', encoding='utf-8') as handle:
    handle.write(json.dumps(capability_row, sort_keys=True) + '\n')
print(capabilities)

## A3 — central-inference stage comparison

This fixed production recipe compares the default one-CUDA-owner lane against explicit direct batching. Both lanes use the same checkpoint, Syzygy files, games, concurrency, and simulations. Python fallback is intentionally not forced because a 128-game Python lane is not a useful T4 production measurement.

In [ ]:
# 7. Fixed central-inference comparison: workers=2, concurrency=128, 128 games, 150 sims.
comparison_base = [
    'python', 'scripts/bench_throughput.py',
    '--checkpoint', str(CHECKPOINT),
    '--output', str(OUTPUT),
    '--device', 'cuda',
    '--workers', '2',
    '--concurrency', '128',
    '--games', '128',
    '--sims', '150',
    '--max-moves', '200',
    '--warmups', '1',
    '--repeats', '3',
    '--syzygy-path', str(SYZYGY),
]

subprocess.run(comparison_base + ['--central-inference', 'on'], check=True)
direct_env = os.environ.copy()
direct_env['IMMORTALITE_ONE_CENTRAL_INFERENCE'] = '0'
subprocess.run(
    comparison_base + ['--central-inference', 'off'],
    check=True,
    env=direct_env,
)
print('central and direct rows appended to', OUTPUT)